# Module 2 — Sentiment / Emotion Classifier

BiLSTM over `dair-ai/emotion`, collapsed from 6 fine emotions into 3 routing buckets (negative / neutral / positive). See `src/config.EMOTION_TO_BUCKET` for the exact mapping and rationale.

**Domain-shift caveat** (documented, not hidden): this dataset is Twitter text, not customer-support text. Treat absolute accuracy numbers here as a routing-quality proxy, not a guarantee of production behavior — the README covers the mitigation.

In [ ]:
import sys
sys.path.append('..')
from datasets import load_dataset
import pandas as pd
from src import config
from src.sentiment.train import EMOTION_NAMES, to_bucket, build_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

## Load & inspect

In [ ]:
ds = load_dataset(config.SENTIMENT_DATASET)
train, val, test = ds['train'], ds['validation'], ds['test']
pd.Series([EMOTION_NAMES[l] for l in train['label']]).value_counts()

## Bucket mapping check

In [ ]:
bucket_counts = pd.Series([config.SENTIMENT_LABELS[to_bucket(l)] for l in train['label']])
bucket_counts.value_counts()

## Tokenize

In [ ]:
tokenizer = Tokenizer(num_words=config.SENTIMENT_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train['text'])

def vectorize(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=config.SENTIMENT_MAX_LEN, padding='post', truncating='post')

X_train, y_train = vectorize(train['text']), np.array([to_bucket(l) for l in train['label']])
X_val, y_val = vectorize(val['text']), np.array([to_bucket(l) for l in val['label']])
X_test, y_test = vectorize(test['text']), np.array([to_bucket(l) for l in test['label']])

## Train

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

model = build_model(vocab_size=config.SENTIMENT_VOCAB_SIZE)
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=15, batch_size=64, class_weight=class_weight_dict,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)],
    verbose=2,
)

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

## Evaluate

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

test_preds = model.predict(X_test).argmax(axis=1)
print(classification_report(y_test, test_preds, target_names=config.SENTIMENT_LABELS))
ConfusionMatrixDisplay.from_predictions(y_test, test_preds, display_labels=config.SENTIMENT_LABELS)
plt.title('Sentiment — confusion matrix')
plt.show()

## Save

In [ ]:
import joblib
model.save(config.SENTIMENT_MODEL_PATH)
joblib.dump(tokenizer, config.SENTIMENT_TOKENIZER_PATH)
print('Saved.')

## Spot-check

In [ ]:
from src.sentiment.predict import predict_sentiment
for s in ["This is the third time my order hasn't arrived, I'm furious.",
          'Just checking on my delivery date.',
          'Thanks so much, that fixed it!']:
    print(s, '->', predict_sentiment(s))